# 🔊 Border Intrusion Detection — Exploratory Data Analysis

**Edge AI Acoustic Classification System**

This notebook performs comprehensive EDA on the audio dataset used for training the border intrusion detection model.

**Classes:**
- 🚶 **Footstep** — Human footstep sounds
- 💥 **Gunshot (Ballistic)** — Firearm discharge sounds  
- 🌿 **Noise** — Environmental sounds (rain, birds, wind, thunder, etc.)

---

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Dark theme for military aesthetic
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0a0e14',
    'axes.facecolor': '#131920',
    'axes.edgecolor': '#39FF14',
    'axes.labelcolor': '#c5c8c6',
    'text.color': '#c5c8c6',
    'xtick.color': '#c5c8c6',
    'ytick.color': '#c5c8c6',
    'grid.color': '#1a2332',
    'font.family': 'monospace',
    'font.size': 11,
})

COLORS = {'footstep': '#39FF14', 'gunshot': '#FF3131', 'noise': '#FFB000'}

DATASET_DIR = os.path.join('..', 'dataset')
SAMPLE_RATE = 22050

print('✅ Libraries loaded')

## 1. Dataset Overview

In [ ]:
# Count files per class
classes = {'footsteps': 'Footstep', 'balastic': 'Gunshot', 'noise': 'Noise'}
file_counts = {}
all_files = {}

for folder, label in classes.items():
    path = os.path.join(DATASET_DIR, folder)
    files = glob.glob(os.path.join(path, '*.wav'))
    file_counts[label] = len(files)
    all_files[label] = files
    print(f'{label:12s}: {len(files):4d} files  ({path})')

print(f'{"TOTAL":12s}: {sum(file_counts.values()):4d} files')

In [ ]:
# Class Distribution Bar Chart
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(file_counts.keys(), file_counts.values(), 
              color=[COLORS['footstep'], COLORS['gunshot'], COLORS['noise']],
              edgecolor='white', linewidth=0.5, alpha=0.9)

for bar, count in zip(bars, file_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            str(count), ha='center', va='bottom', fontweight='bold', fontsize=14, color='#39FF14')

ax.set_title('CLASS DISTRIBUTION — AUDIO SAMPLES', fontsize=16, fontweight='bold', color='#39FF14')
ax.set_ylabel('Number of Samples', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0a0e14')
plt.show()

## 2. Audio Properties Analysis

In [ ]:
# Analyze audio properties
import soundfile as sf

audio_props = []
for label, files in all_files.items():
    for f in files[:50]:  # sample first 50 per class
        try:
            info = sf.info(f)
            audio, sr = librosa.load(f, sr=SAMPLE_RATE)
            audio_props.append({
                'class': label,
                'duration_s': info.duration,
                'sample_rate': info.samplerate,
                'channels': info.channels,
                'rms_energy': float(np.sqrt(np.mean(audio**2))),
                'peak_amplitude': float(np.max(np.abs(audio))),
                'zero_crossing_rate': float(np.mean(librosa.feature.zero_crossing_rate(audio))),
                'spectral_centroid': float(np.mean(librosa.feature.spectral_centroid(y=audio, sr=SAMPLE_RATE))),
            })
        except Exception as e:
            pass

df_props = pd.DataFrame(audio_props)
print(df_props.groupby('class').describe().round(4))

In [ ]:
# Audio Properties Distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['rms_energy', 'peak_amplitude', 'zero_crossing_rate', 'spectral_centroid']
titles = ['RMS Energy', 'Peak Amplitude', 'Zero Crossing Rate', 'Spectral Centroid (Hz)']

for ax, metric, title in zip(axes.flat, metrics, titles):
    for label in ['Footstep', 'Gunshot', 'Noise']:
        data = df_props[df_props['class'] == label][metric]
        color_key = label.lower()
        ax.hist(data, bins=20, alpha=0.6, label=label, color=COLORS[color_key], edgecolor='white', linewidth=0.3)
    ax.set_title(title, fontweight='bold', color='#39FF14')
    ax.legend(facecolor='#131920', edgecolor='#39FF14')
    ax.grid(alpha=0.2)

plt.suptitle('AUDIO FEATURE DISTRIBUTIONS', fontsize=16, fontweight='bold', color='#39FF14', y=1.02)
plt.tight_layout()
plt.savefig('../results/audio_distributions.png', dpi=150, bbox_inches='tight', facecolor='#0a0e14')
plt.show()

## 3. Waveform Visualization

In [ ]:
# Sample waveforms for each class
fig, axes = plt.subplots(3, 3, figsize=(18, 10))

for row, (label, files) in enumerate(all_files.items()):
    color_key = label.lower()
    for col in range(3):
        audio, sr = librosa.load(files[col], sr=SAMPLE_RATE)
        t = np.linspace(0, len(audio)/sr, len(audio))
        axes[row, col].plot(t, audio, color=COLORS[color_key], linewidth=0.5, alpha=0.8)
        axes[row, col].fill_between(t, audio, alpha=0.2, color=COLORS[color_key])
        axes[row, col].set_title(f'{label} — Sample {col+1}', fontweight='bold', color=COLORS[color_key])
        axes[row, col].set_ylim(-1, 1)
        axes[row, col].set_xlabel('Time (s)')
        if col == 0:
            axes[row, col].set_ylabel('Amplitude')
        axes[row, col].grid(alpha=0.2)

plt.suptitle('WAVEFORM SAMPLES PER CLASS', fontsize=16, fontweight='bold', color='#39FF14', y=1.02)
plt.tight_layout()
plt.savefig('../results/waveforms.png', dpi=150, bbox_inches='tight', facecolor='#0a0e14')
plt.show()

## 4. Spectrogram Analysis

In [ ]:
# Mel Spectrograms
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (label, files) in enumerate(all_files.items()):
    audio, sr = librosa.load(files[0], sr=SAMPLE_RATE)
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    img = librosa.display.specshow(mel_spec_db, sr=sr, x_axis='time', y_axis='mel', 
                                   ax=axes[idx], cmap='inferno')
    color_key = label.lower()
    axes[idx].set_title(f'{label} — Mel Spectrogram', fontweight='bold', color=COLORS[color_key])
    fig.colorbar(img, ax=axes[idx], format='%+2.0f dB')

plt.suptitle('MEL SPECTROGRAM COMPARISON', fontsize=16, fontweight='bold', color='#39FF14', y=1.02)
plt.tight_layout()
plt.savefig('../results/mel_spectrograms.png', dpi=150, bbox_inches='tight', facecolor='#0a0e14')
plt.show()

## 5. MFCC Feature Analysis

In [ ]:
# MFCC Heatmaps
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (label, files) in enumerate(all_files.items()):
    audio, sr = librosa.load(files[0], sr=SAMPLE_RATE)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    
    img = librosa.display.specshow(mfcc, sr=sr, x_axis='time', ax=axes[idx], cmap='coolwarm')
    color_key = label.lower()
    axes[idx].set_title(f'{label} — MFCC Features', fontweight='bold', color=COLORS[color_key])
    axes[idx].set_ylabel('MFCC Coefficient')
    fig.colorbar(img, ax=axes[idx])

plt.suptitle('MFCC FEATURE HEATMAPS', fontsize=16, fontweight='bold', color='#39FF14', y=1.02)
plt.tight_layout()
plt.savefig('../results/mfcc_heatmaps.png', dpi=150, bbox_inches='tight', facecolor='#0a0e14')
plt.show()

## 6. MFCC Embedding Visualization (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

# Extract MFCC means for all samples
mfcc_means = []
labels = []
label_map = {'footsteps': 'Footstep', 'balastic': 'Gunshot', 'noise': 'Noise'}

for folder, label in label_map.items():
    path = os.path.join(DATASET_DIR, folder)
    files = glob.glob(os.path.join(path, '*.wav'))
    for f in files[:60]:  # max 60 per class for speed
        try:
            audio, sr = librosa.load(f, sr=SAMPLE_RATE, duration=1.0)
            mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
            mfcc_means.append(np.mean(mfcc, axis=1))
            labels.append(label)
        except:
            pass

X_mfcc = np.array(mfcc_means)
print(f'Extracted MFCC embeddings: {X_mfcc.shape}')

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_embedded = tsne.fit_transform(X_mfcc)

fig, ax = plt.subplots(figsize=(10, 8))
for label in ['Footstep', 'Gunshot', 'Noise']:
    mask = np.array(labels) == label
    color_key = label.lower()
    ax.scatter(X_embedded[mask, 0], X_embedded[mask, 1], 
               c=COLORS[color_key], label=label, alpha=0.7, s=60, edgecolors='white', linewidth=0.3)

ax.set_title('t-SNE VISUALIZATION OF MFCC EMBEDDINGS', fontsize=16, fontweight='bold', color='#39FF14')
ax.legend(facecolor='#131920', edgecolor='#39FF14', fontsize=12)
ax.grid(alpha=0.15)
plt.tight_layout()
plt.savefig('../results/tsne_mfcc.png', dpi=150, bbox_inches='tight', facecolor='#0a0e14')
plt.show()

## 7. Summary Statistics

In [ ]:
# Final Summary
print('=' * 60)
print('DATASET SUMMARY — BORDER INTRUSION DETECTION')
print('=' * 60)
print(f'\nTotal samples: {sum(file_counts.values())}')
for label, count in file_counts.items():
    pct = count / sum(file_counts.values()) * 100
    print(f'  {label:12s}: {count:4d} ({pct:.1f}%)')
print(f'\nSample rate: {SAMPLE_RATE} Hz')
print(f'Features: 40 MFCCs, 128 Mel bands')
print(f'\n⚠ Class imbalance detected:')
print(f'  Gunshot has {file_counts["Gunshot"]/file_counts["Footstep"]:.1f}x more samples than Footstep')
print(f'  → Data augmentation required for Footstep class')
print('=' * 60)